In [1]:
import math

# 实现余弦缩放奖励函数
def get_cosine_scaled_reward(
    # 错误答案的最小奖励值，默认为 -0.5
    min_value_wrong: float = -0.5,
    # 错误答案的最大奖励值，默认为 -0.1
    max_value_wrong: float = -0.1,
    # 正确答案的最小奖励值，默认为 0.8
    min_value_correct: float = 0.8,
    # 正确答案的最大奖励值，默认为 1.0
    max_value_correct: float = 1.0,
    # 最大长度，用于计算进度，默认为 1000
    max_len: int = 1000,
):
    """
    返回一个余弦缩放奖励函数。此函数会根据完成长度对准确率奖励进行缩放。
    较短的正确答案会获得更高的奖励，较长的错误答案会减少惩罚。
    """
    def cosine_scaled_reward(completions, solution, accuracy_rewards, **kwargs):
        """
        余弦缩放奖励函数，根据完成长度调整准确率奖励。
        """
        # 从每个完成项中提取内容
        contents = [completion[0]["content"] for completion in completions]
        # 用于存储每个完成项的最终奖励值
        rewards = []

        # 遍历每个完成项的内容、对应的标准答案和准确率奖励
        for content, sol, acc_reward in zip(contents, solution, accuracy_rewards):
            # 生成答案的长度
            gen_len = len(content)
            # 计算当前生成答案长度相对于最大长度的进度
            progress = gen_len / max_len
            # 根据进度计算余弦值
            cosine = math.cos(progress * math.pi)

            # 判断答案是否正确
            if acc_reward > 0.5:  # 假设准确率奖励对于正确答案约为 1.0
                # 正确答案的最小奖励值
                min_value = min_value_correct
                # 正确答案的最大奖励值
                max_value = max_value_correct
            else:  # 错误答案
                # 注意这里的交换，错误答案的最小惩罚值（实际是最大奖励值）
                min_value = max_value_wrong
                # 错误答案的最大惩罚值（实际是最小奖励值）
                max_value = min_value_wrong

            # 余弦缩放公式，根据余弦值和最大最小奖励值计算最终奖励
            reward = min_value + 0.5 * (max_value - min_value) * (1.0 + cosine)
            # 将计算得到的奖励值添加到奖励列表中
            rewards.append(float(reward))
        # 返回所有完成项的最终奖励值列表
        return rewards
    # 返回内部定义的余弦缩放奖励函数
    return cosine_scaled_reward